# Train a YOLOv8 License Plate Detector (Google Colab)

**What this notebook does:** takes a labeled license-plate image dataset, trains a YOLOv8 object detector on it, checks how good it is, and gives you the file **`best.pt`**. The web app (`app.py`) loads that file.

**How YOLO works in one minute:** YOLO ("You Only Look Once") is a neural network that looks at an image once and outputs boxes around the objects it was trained to find, each with a confidence score. We start from `yolov8n.pt` (a small model already trained on everyday objects) and *fine-tune* it on our plate images so it learns what a license plate looks like. This is called transfer learning and it is why a small dataset and ~30 minutes of training are enough.

**Before you run anything:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

## 1. Setup

In [ ]:
%pip install -q ultralytics
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Get the dataset into Colab
Run the cell below. A **Choose files** button appears: pick the dataset **zip** from your computer (it uploads straight to Colab, so no Google Drive access is needed). Wait until it says 100%.

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose the dataset zip from your computer
DATASET_ZIP = "/content/" + list(uploaded.keys())[0]
print("Uploaded:", DATASET_ZIP)

import zipfile, os
os.makedirs("/content/dataset", exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall("/content/dataset")

# Look at what is inside
for root, dirs, files in os.walk("/content/dataset"):
    depth = root.replace("/content/dataset", "").count(os.sep)
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)")

## 3. Put the data in the layout YOLO expects
YOLO wants `images/train`, `images/val`, `labels/train`, `labels/val`, where each label file has one line per plate: `class x_center y_center width height` (all between 0 and 1).

The cell below works with the two common dataset formats: **YOLO `.txt` labels** and **Pascal VOC `.xml` labels** (it converts these). It pairs every image with its label, makes an 85/15 train/validation split, and treats everything as one class: `license_plate`.

If your dataset has other classes too (like "vehicle"), set `PLATE_CLASS_IDS` to the id(s) of the plate class(es) after you've seen the printout.

In [ ]:
import random, shutil, xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

SRC = Path("/content/dataset")
OUT = Path("/content/plates_yolo")
PLATE_CLASS_IDS = None      # None = keep every box. Or e.g. {1} to keep only class id 1.
random.seed(42)

IMG_EXT = {".jpg", ".jpeg", ".png"}
images = {p.stem: p for p in SRC.rglob("*") if p.suffix.lower() in IMG_EXT}
labels = {p.stem: p for p in SRC.rglob("*")
          if p.suffix.lower() in {".txt", ".xml"} and p.stem.lower() not in {"classes", "readme", "labels"}}

pairs = [(images[s], labels[s]) for s in images if s in labels]
print(f"{len(images)} images, {len(labels)} label files, {len(pairs)} matched pairs")
assert pairs, "No image/label pairs found. Check the folder printout above."

def read_boxes(label_path, img_path):
    """Return [(class_id, xc, yc, w, h)] in normalised YOLO format."""
    boxes = []
    if label_path.suffix == ".txt":
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                boxes.append((int(float(parts[0])), *map(float, parts[1:5])))
    else:  # Pascal VOC xml
        root = ET.parse(label_path).getroot()
        W = float(root.find("size/width").text); H = float(root.find("size/height").text)
        for obj in root.iter("object"):
            b = obj.find("bndbox")
            x1, y1, x2, y2 = (float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax"))
            boxes.append((0, (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H))
    return boxes

random.shuffle(pairs)
split = int(len(pairs) * 0.85)
sets = {"train": pairs[:split], "val": pairs[split:]}

class_counts, kept = Counter(), 0
for name, items in sets.items():
    (OUT / "images" / name).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / name).mkdir(parents=True, exist_ok=True)
    for img, lab in items:
        boxes = read_boxes(lab, img)
        for b in boxes: class_counts[b[0]] += 1
        if PLATE_CLASS_IDS is not None:
            boxes = [b for b in boxes if b[0] in PLATE_CLASS_IDS]
        if not boxes:
            continue
        shutil.copy(img, OUT / "images" / name / img.name)
        (OUT / "labels" / name / f"{img.stem}.txt").write_text(
            "\n".join(f"0 {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}" for b in boxes))
        kept += 1

print("Class ids found in the labels:", dict(class_counts))
print(f"Usable images: {kept}  (train {len(list((OUT/'images'/'train').iterdir()))}, "
      f"val {len(list((OUT/'images'/'val').iterdir()))})")

(OUT / "data.yaml").write_text(f"path: {OUT}\ntrain: images/train\nval: images/val\nnames:\n  0: license_plate\n")
print((OUT / "data.yaml").read_text())

## 4. Sanity check: are the boxes in the right place?
Always look at a few labeled images before training. If the boxes don't sit on the plates, the labels were read wrongly and training would be useless.

In [ ]:
import cv2, matplotlib.pyplot as plt
sample = random.sample(list((OUT / "images" / "train").iterdir()), 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, p in zip(axes.ravel(), sample):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for line in (OUT / "labels" / "train" / f"{p.stem}.txt").read_text().splitlines():
        _, xc, yc, bw, bh = map(float, line.split())
        x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
        x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train
- `yolov8n.pt`: the small, fast pretrained starting point.
- `epochs`: how many passes over the training images. `patience=15` stops early if the model stops improving.
- `imgsz=640`: images are resized to 640 px.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=str(OUT / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="/content/runs",
    name="plates",
    seed=42,
)

## 6. Evaluate on the validation images
- **Precision:** of the plates the model drew a box on, how many really were plates.
- **Recall:** of the real plates, how many the model found.
- **mAP50:** overall detection quality counting a box as correct if it overlaps the true plate by at least 50% (higher is better; above about 0.9 is very good for a single-class task).

In [ ]:
best = YOLO("/content/runs/plates/weights/best.pt")
m = best.val(data=str(OUT / "data.yaml"))
print(f"Precision: {m.box.mp:.3f}")
print(f"Recall:    {m.box.mr:.3f}")
print(f"mAP50:     {m.box.map50:.3f}")
print(f"mAP50-95:  {m.box.map:.3f}")

In [ ]:
from IPython.display import Image, display
for name in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    p = f"/content/runs/plates/{name}"
    if os.path.exists(p):
        display(Image(p, width=800))

## 7. See it work on validation images

In [ ]:
val_imgs = random.sample(list((OUT / "images" / "val").iterdir()), min(6, len(list((OUT / "images" / "val").iterdir()))))
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, p in zip(axes.ravel(), val_imgs):
    r = best.predict(str(p), conf=0.35, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); plt.show()

## 8. Save the model
This downloads `best.pt` to your computer. Put it in the project at `model/best.pt`. (Colab deletes its files when the session ends, so make sure the download finished.)

In [ ]:
from google.colab import files
files.download("/content/runs/plates/weights/best.pt")   # saves best.pt to your computer

## 9. (Optional) Test on a video
Upload a short video to Colab (left sidebar → Files), set the path, run. The annotated video is saved under `/content/runs/`.

In [ ]:
VIDEO = "/content/test_video.mp4"   # <-- change me
if os.path.exists(VIDEO):
    best.predict(VIDEO, conf=0.35, save=True, project="/content/runs", name="video_test")
    print("Done. Look in /content/runs/video_test/")
else:
    print("Upload a video first, or skip this cell.")